In [ ]:
import pandas as pd
import pyodbc
import boto3
from io import BytesIO
import datetime

## Configuration

In [ ]:
SQL_CONN_STR = (
    "DRIVER={ODBC Driver 18 for SQL Server};" #or equivalent ODBC version you have installed
    "SERVER=<server_name and port>"
    "DATABASE=<database_name>;" #the database name you're using
    "UID=<username>;"
    "PWD=<password>;" 
    "TrustServerCertificate=yes;"
)

S3_BUCKET = "<bucket_name>"

# Legacy tables to migrate and modernize 
TABLES_TO_EXTRACT = [ 
    {"schema": "Sales", "table": "SalesOrderHeader"},
    {"schema": "Sales", "table": "Customer"},
    {"schema": "Person", "table": "Person"}
]

### Initialize S3 Client

In [ ]:
s3_client = boto3.client('s3')

def extract_to_s3():
    try:
        print("Connecting to SQL Server...")
        conn = pyodbc.connect(SQL_CONN_STR)
        
        for item in TABLES_TO_EXTRACT:
            source_name = f"{item['schema']}.{item['table']}"
            target_key = f"raw/{item['table'].lower()}/{item['table'].lower()}.parquet"
            
            print(f"Reading {source_name}...")
            df = pd.read_sql(f"SELECT * FROM {source_name}", conn)

    
            # Floor the data in the DataFrame to Milliseconds (3 decimals)
            for col in df.select_dtypes(include=['datetime64']).columns:
                df[col] = df[col].dt.floor('ms')

            # Convert to Parquet forcing the 'ms' resolution in metadata
            print(f"Converting {item['table']} to Parquet (Resolution: ms)...")
            parquet_buffer = BytesIO()
            df.to_parquet(
                parquet_buffer, 
                index=False, 
                engine='pyarrow',
                coerce_timestamps='ms',         # Forces 3-decimal storage
                allow_truncated_timestamps=True # Prevents overflow errors
            )

            # Upload to S3
            print(f"Uploading to s3://{S3_BUCKET}/{target_key}...")
            s3_client.put_object(
                Bucket=S3_BUCKET,
                Key=target_key,
                Body=parquet_buffer.getvalue()
            )
            
            print(f"✅ {item['table']} successfully landed in S3.\n")

    except Exception as e:
        print(f"❌ Pipeline failed: {e}")
    finally:
        if 'conn' in locals():
            conn.close()

if __name__ == "__main__":
    extract_to_s3()